INSTALL REQUIRED LIBRARIES

In [1]:
!pip install -q groq gradio pypdf python-docx plotly pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.0/346.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.4 MB/s eta 0:00:00


In [2]:
pip install --upgrade gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: starlette
    Found existing installation: starlette 0.52.1
    Uninstalling starlette-0.52.1:
      Successfully uninstalled starlette-0.52.1
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.2.1 whi

Import data handling and visualization libraries

In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

SET UP GROQ API CONNECTION


In [4]:
# Import required modules
import json
import re
from pypdf import PdfReader          # For reading PDF resumes
from docx import Document            # For reading DOCX resumes
import gradio as gr                  # For building UI

from google.colab import userdata
from groq import Groq               # Groq API client

# Fetch API key securely from Colab
groq_api_key = userdata.get("GROQ_API_KEY")
# Initialize Groq client
client = Groq(api_key=groq_api_key)

In [5]:
# Test API connection
try:
    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "user", "content": "Reply in one line: Groq API is connected successfully."}
        ],
        model="llama-3.1-8b-instant"
    )
    print(chat_completion.choices[0].message.content)
except Exception as e:
    print("Groq test failed:")
    print(e)

Verification of the Groq API connection is complete, proceeding with API operations.


In [6]:
# COMPANY & ROLE CONFIGURATION


# Classify companies into product-based vs service-based
COMPANY_TYPE_MAP = {
    # Product-based (FAANG + major tech)
    "Google": "product",
    "Microsoft": "product",
    "Amazon": "product",
    "Meta": "product",
    "Apple": "product",
    "Netflix": "product",
    "Flipkart": "product",
    "Zomato": "product",
    "Swiggy": "product",
    "Paytm": "product",
    "Razorpay": "product",
    "Atlassian": "product",
    "Adobe": "product",
    "Salesforce": "product",
    "Uber": "product",
    "Airbnb": "product",
    "Zscaler": "product",
    "ServiceNow": "product",
    "SAP": "product",
    "Oracle": "product",
    # Service-based
    "TCS": "service",
    "Infosys": "service",
    "Wipro": "service",
    "HCL": "service",
    "Cognizant": "service",
    "Capgemini": "service",
    "Accenture": "service",
    "Tech Mahindra": "service",
    "Mphasis": "service",
    "LTIMindtree": "service",
    "Hexaware": "service",
    "Persistent Systems": "service",
    "KPIT": "service",
}

In [7]:
# Topics per role for PRODUCT-based companies (harder DSA, system design, fundamentals)
PRODUCT_ROLE_TOPICS = {
    "Software Engineer": {
        "primary": ["Hard DSA", "Dynamic Programming", "Graphs & Trees", "System Design", "OOP"],
        "secondary": ["Operating System", "Computer Networks", "DBMS", "SQL (light)"],
        "dsa_level": "hard",
        "sql_weight": "light"
    },
    "SDE": {
        "primary": ["Hard DSA", "Dynamic Programming", "Graphs & Trees", "System Design", "OOP"],
        "secondary": ["OS", "CN", "DBMS", "Multithreading"],
        "dsa_level": "hard",
        "sql_weight": "light"
    },
    "Data Analyst": {
        "primary": ["Advanced SQL", "Statistics", "Data Cleaning", "Python/Pandas", "Data Visualization"],
        "secondary": ["Machine Learning basics", "DBMS", "Excel/Power BI", "Case Studies"],
        "dsa_level": "basic",
        "sql_weight": "heavy"
    },
    "Data Scientist": {
        "primary": ["Machine Learning", "Statistics", "Feature Engineering", "Model Evaluation", "Python"],
        "secondary": ["Advanced SQL", "Deep Learning basics", "Data Wrangling", "NumPy"],
        "dsa_level": "basic",
        "sql_weight": "medium"
    },
    "Machine Learning Engineer": {
        "primary": ["Machine Learning algorithms", "Deep Learning", "Model Deployment", "Python", "MLOps"],
        "secondary": ["System Design for ML", "SQL", "Feature Engineering", "Cloud/AWS/GCP"],
        "dsa_level": "medium",
        "sql_weight": "medium"
    },
    "Frontend Developer": {
        "primary": ["JavaScript (advanced)", "React", "UI System Design", "Performance Optimization", "CSS/HTML"],
        "secondary": ["Medium DSA", "APIs", "Security (XSS/CSRF)", "Browser internals"],
        "dsa_level": "medium",
        "sql_weight": "none"
    },
    "Backend Developer": {
        "primary": ["System Design", "APIs", "Database Optimization", "Caching", "Scalability"],
        "secondary": ["Hard DSA", "Security", "Microservices", "Message Queues"],
        "dsa_level": "hard",
        "sql_weight": "medium"
    },
    "DevOps Engineer": {
        "primary": ["CI/CD pipelines", "Docker & Kubernetes", "Cloud (AWS/GCP/Azure)", "Linux/Bash scripting", "Monitoring"],
        "secondary": ["System Design", "Networking", "Security", "Infrastructure as Code"],
        "dsa_level": "basic",
        "sql_weight": "light"
    },
    "Web Developer": {
        "primary": ["JavaScript", "React/Vue/Angular", "REST APIs", "CSS/HTML", "Performance"],
        "secondary": ["Medium DSA", "System Design (UI)", "Security"],
        "dsa_level": "medium",
        "sql_weight": "light"
    },
    "Java Developer": {
        "primary": ["Java Core", "OOP", "Multithreading", "Spring Boot", "Collections Framework"],
        "secondary": ["DSA (medium)", "DBMS/JPA", "Design Patterns", "REST APIs"],
        "dsa_level": "medium",
        "sql_weight": "medium"
    },
    "Python Developer": {
        "primary": ["Python advanced", "OOP", "Flask/Django", "APIs", "Libraries (NumPy, Pandas)"],
        "secondary": ["DSA (medium)", "DBMS", "Testing", "Async programming"],
        "dsa_level": "medium",
        "sql_weight": "medium"
    },
    "QA Engineer": {
        "primary": ["Test Design", "Automation (Selenium/Cypress)", "API Testing", "Bug Reporting", "CI/CD"],
        "secondary": ["Basic DSA", "SQL", "Performance Testing"],
        "dsa_level": "basic",
        "sql_weight": "medium"
    },
    "Business Analyst": {
        "primary": ["SQL", "Requirement Gathering", "Data Analysis", "Process Modeling", "Excel"],
        "secondary": ["Basic DBMS", "Agile/Scrum", "Communication", "Reporting tools"],
        "dsa_level": "none",
        "sql_weight": "heavy"
    },
}

In [8]:
# Topics per role for SERVICE-based companies (simpler DSA, SQL/DBMS heavy, aptitude)
SERVICE_ROLE_TOPICS = {
    "Software Engineer": {
        "primary": ["SQL & DBMS", "Basic DSA (Arrays, Strings, Sorting)", "OOP concepts", "Core Java/Python basics"],
        "secondary": ["OS basics", "CN basics", "Aptitude patterns", "Verbal/Logical reasoning patterns"],
        "dsa_level": "basic",
        "sql_weight": "heavy"
    },
    "SDE": {
        "primary": ["SQL & DBMS", "Basic DSA", "OOP", "Coding patterns (loops, recursion)"],
        "secondary": ["OS", "CN", "Aptitude"],
        "dsa_level": "basic",
        "sql_weight": "heavy"
    },
    "Data Analyst": {
        "primary": ["SQL (Heavy - Joins, Subqueries, Window Functions)", "DBMS", "Excel", "Power BI/Tableau"],
        "secondary": ["Python basics", "Data Cleaning", "Reporting", "MS Excel formulas"],
        "dsa_level": "none",
        "sql_weight": "heavy"
    },
    "Data Scientist": {
        "primary": ["SQL", "Python basics", "ML Concepts (high level)", "Statistics basics"],
        "secondary": ["DBMS", "Excel", "Data Visualization", "Reporting"],
        "dsa_level": "basic",
        "sql_weight": "heavy"
    },
    "Machine Learning Engineer": {
        "primary": ["ML algorithms (conceptual)", "Python", "SQL", "Data preprocessing"],
        "secondary": ["Statistics", "Basic DSA", "DBMS"],
        "dsa_level": "basic",
        "sql_weight": "medium"
    },
    "Frontend Developer": {
        "primary": ["HTML/CSS", "JavaScript basics", "React (basic)", "Responsive Design"],
        "secondary": ["Basic DSA", "REST APIs"],
        "dsa_level": "basic",
        "sql_weight": "light"
    },
    "Backend Developer": {
        "primary": ["SQL & DBMS", "APIs (REST basics)", "OOP", "Basic DSA"],
        "secondary": ["OS", "CN", "Java/Python basics"],
        "dsa_level": "basic",
        "sql_weight": "heavy"
    },
    "DevOps Engineer": {
        "primary": ["Linux basics", "CI/CD concepts", "Docker basics", "Cloud fundamentals"],
        "secondary": ["Networking basics", "Scripting (Bash/Python)"],
        "dsa_level": "none",
        "sql_weight": "light"
    },
    "Web Developer": {
        "primary": ["HTML/CSS/JavaScript", "SQL", "Basic Backend (PHP/Node basics)", "REST APIs"],
        "secondary": ["Basic DSA", "DBMS"],
        "dsa_level": "basic",
        "sql_weight": "medium"
    },
    "Java Developer": {
        "primary": ["Core Java", "OOP", "Collections", "SQL/JDBC", "Exception Handling"],
        "secondary": ["Basic DSA", "Spring Boot basics", "DBMS"],
        "dsa_level": "basic",
        "sql_weight": "heavy"
    },
    "Python Developer": {
        "primary": ["Python basics", "OOP", "SQL", "File Handling", "Flask/Django basics"],
        "secondary": ["Basic DSA", "DBMS"],
        "dsa_level": "basic",
        "sql_weight": "heavy"
    },
    "QA Engineer": {
        "primary": ["Manual Testing", "Test Cases", "SQL", "Bug Lifecycle", "STLC/SDLC"],
        "secondary": ["Basic Automation", "Agile"],
        "dsa_level": "none",
        "sql_weight": "medium"
    },
    "Business Analyst": {
        "primary": ["SQL", "DBMS", "Excel", "Requirement Analysis", "Communication"],
        "secondary": ["Agile/Scrum", "Reporting Tools", "Process Flows"],
        "dsa_level": "none",
        "sql_weight": "heavy"
    },
}

In [9]:
# Fallback default topics when role is not in map
DEFAULT_PRODUCT_TOPICS = {
    "primary": ["DSA (medium-hard)", "System Design", "OOP", "Computer Science fundamentals"],
    "secondary": ["SQL", "OS", "CN"],
    "dsa_level": "medium",
    "sql_weight": "medium"
}

DEFAULT_SERVICE_TOPICS = {
    "primary": ["SQL & DBMS", "Basic DSA", "OOP", "Core Programming concepts"],
    "secondary": ["OS basics", "CN basics", "Aptitude"],
    "dsa_level": "basic",
    "sql_weight": "heavy"
}

In [10]:
# Dropdown lists for UI
PRODUCT_COMPANIES = [
    "Google", "Microsoft", "Amazon", "Meta", "Apple", "Netflix",
    "Flipkart", "Zomato", "Swiggy", "Paytm", "Razorpay", "Atlassian",
    "Adobe", "Salesforce", "Uber", "Airbnb", "Zscaler", "ServiceNow",
    "SAP", "Oracle"
]

SERVICE_COMPANIES = [
    "TCS", "Infosys", "Wipro", "HCL", "Cognizant", "Capgemini",
    "Accenture", "Tech Mahindra", "Mphasis", "LTIMindtree",
    "Hexaware", "Persistent Systems", "KPIT"
]

ALL_COMPANIES = ["(Not specified)"] + sorted(PRODUCT_COMPANIES) + sorted(SERVICE_COMPANIES)

ALL_ROLES = [
    "Software Engineer",
    "SDE",
    "Data Analyst",
    "Data Scientist",
    "Machine Learning Engineer",
    "Frontend Developer",
    "Backend Developer",
    "Web Developer",
    "Java Developer",
    "Python Developer",
    "DevOps Engineer",
    "QA Engineer",
    "Business Analyst"
]

In [11]:

# HELPER: Get question config based on company + role


def get_question_config(role, company):
    """
    Returns topic config dict based on:
    - Company type (product / service / unknown)
    - Job role
    """
    company = (company or "").strip()
    role = (role or "Software Engineer").strip()

    company_type = COMPANY_TYPE_MAP.get(company, "unknown")

    if company_type == "product":
        config = PRODUCT_ROLE_TOPICS.get(role, DEFAULT_PRODUCT_TOPICS)
        config["company_type"] = "product"
    elif company_type == "service":
        config = SERVICE_ROLE_TOPICS.get(role, DEFAULT_SERVICE_TOPICS)
        config["company_type"] = "service"
    else:
        # No company specified or unknown — use product topics as default
        config = PRODUCT_ROLE_TOPICS.get(role, DEFAULT_PRODUCT_TOPICS)
        config["company_type"] = "general"

    config["role"] = role
    config["company"] = company if company else "Not specified"
    return config


In [12]:
# RESUME TEXT EXTRACTION

def extract_text_from_pdf(file_path):
    text = ""
    reader = PdfReader(file_path)
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text.strip()


def extract_text_from_docx(file_path):
    doc = Document(file_path)
    return "\n".join([para.text for para in doc.paragraphs]).strip()


def extract_resume_text(file_path):
    file_path = str(file_path)
    if file_path.endswith(".pdf"):
        return extract_text_from_pdf(file_path)
    elif file_path.endswith(".docx"):
        return extract_text_from_docx(file_path)
    else:
        return "Unsupported file format"


def clean_resume_text(text):
    text = re.sub(r'(\b[A-Za-z]\s){3,}[A-Za-z]\b', lambda m: m.group(0).replace(" ", ""), text)
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

In [13]:
# LLM HELPER


def ask_llm(prompt, model="llama-3.1-8b-instant", temperature=0.4):
    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "system", "content": "You are a helpful interview preparation AI assistant."},
            {"role": "user", "content": prompt}
        ],
        model=model,
        temperature=temperature
    )
    return chat_completion.choices[0].message.content

In [14]:
# BUILD CANDIDATE PROFILE FROM RESUME


def build_candidate_profile_llm(resume_text):
    prompt = f"""
You are an expert resume parser.

Extract information from the following resume text and return ONLY valid JSON.

Format:
{{
  "name": "",
  "summary": "",
  "skills": [],
  "projects": [
    {{
      "title": "",
      "tech_stack": [],
      "description": ""
    }}
  ],
  "education": [],
  "experience": [],
  "target_roles": [],
  "strengths": []
}}

Resume Text:
\"\"\"{resume_text}\"\"\"
"""
    raw = ask_llm(prompt).strip()
    if raw.startswith("```"):
        raw = raw.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(raw)
    except Exception:
        return {
            "name": "Unknown Candidate",
            "summary": resume_text[:300],
            "skills": [],
            "projects": [],
            "education": [],
            "experience": [],
            "target_roles": [],
            "strengths": []
        }

In [15]:
# GENERATE INTERVIEW QUESTIONS — RESUME BASED
# (uses company + role config to shape questions)


def generate_interview_questions(profile, role, company):
    """
    Generate personalized questions from resume profile,
    shaped by company type and job role.
    """
    config = get_question_config(role, company)

    company_context = ""
    if config["company_type"] == "product":
        company_context = f"""
This is for a PRODUCT-BASED company ({config['company']}).
Focus heavily on: {', '.join(config['primary'])}.
DSA difficulty: {config['dsa_level']} (include Dynamic Programming, Graph problems, Tree problems).
SQL weight: {config['sql_weight']}.
Include system design questions.
"""
    elif config["company_type"] == "service":
        company_context = f"""
This is for a SERVICE-BASED company ({config['company']}).
Focus heavily on: {', '.join(config['primary'])}.
DSA difficulty: {config['dsa_level']} (basic arrays, strings, sorting — no hard DP or graphs).
SQL weight: {config['sql_weight']} — include DBMS, SQL queries, normalization.
Do NOT include advanced system design — keep concepts moderate.
"""
    else:
        company_context = f"""
No specific company mentioned.
Focus on: {', '.join(config['primary'])}.
DSA difficulty: {config['dsa_level']}.
SQL weight: {config['sql_weight']}.
"""

    prompt = f"""
You are an expert technical interviewer for the role of {role}.

{company_context}

Candidate Profile:
{json.dumps(profile, indent=2)}

Generate interview questions tailored to BOTH the candidate's skills/projects AND the role/company requirements above.

Rules:
- Technical questions must match the DSA level and topic priorities defined above.
- Reference the candidate's actual skills and technologies from the profile.
- Project questions must be specific to the candidate's listed projects.
- HR questions should suit a student/fresher.
- Do NOT include answers or descriptions.
- Each question must be a plain string.

Return ONLY valid JSON:
{{
  "technical": ["q1", "q2", "q3", "q4", "q5"],
  "project": ["q1", "q2", "q3"],
  "hr": ["q1", "q2", "q3"]
}}
"""
    raw = ask_llm(prompt).strip()
    if raw.startswith("```"):
        raw = raw.replace("```json", "").replace("```", "").strip()

    try:
        data = json.loads(raw)
        return _normalize_questions(data)
    except Exception:
        return _fallback_questions(role)


In [16]:
# GENERATE INTERVIEW QUESTIONS — ROLE BASED (no resume)
# ============================================================

def generate_role_company_questions(role, company="", level="fresher"):
    """
    Generate questions purely from role + company, no resume needed.
    Shapes difficulty and topics based on company type.
    """
    config = get_question_config(role, company)

    company_note = ""
    if config["company_type"] == "product":
        company_note = f"""
Company type: PRODUCT-BASED ({config['company']})
This company is known for rigorous technical interviews.
Primary topics: {', '.join(config['primary'])}
Secondary topics: {', '.join(config['secondary'])}
DSA level: {config['dsa_level']} — include Dynamic Programming, Graph algorithms, Tree problems, Recursion.
SQL: {config['sql_weight']} weight.
Include 1-2 System Design questions in the technical round.
"""
    elif config["company_type"] == "service":
        company_note = f"""
Company type: SERVICE-BASED ({config['company']})
This company focuses on fundamentals and practical knowledge.
Primary topics: {', '.join(config['primary'])}
Secondary topics: {', '.join(config['secondary'])}
DSA level: {config['dsa_level']} — only basic problems like arrays, strings, simple sorting.
SQL: {config['sql_weight']} weight — include SQL queries, joins, DBMS normalization, indexing.
Do NOT include advanced system design or hard DP problems.
"""
    else:
        company_note = f"""
No specific company mentioned. Use general interview standards for role: {role}.
Primary topics: {', '.join(config['primary'])}
DSA level: {config['dsa_level']}.
SQL: {config['sql_weight']} weight.
"""

    prompt = f"""
You are an expert technical interviewer.

Generate interview questions for:
Role: {role}
Candidate level: {level}

{company_note}

Generate:
1. 5 technical interview questions
2. 3 project/scenario-based interview questions
3. 3 HR interview questions

Rules:
- Questions must be specific to the selected role and topic priorities above.
- If company is product-based, ensure at least 2 hard DSA questions and 1 system design question.
- If company is service-based, focus on SQL, DBMS, and straightforward coding questions.
- HR questions should be suitable for student/fresher level.
- Do NOT include answers.
- Return ONLY valid JSON:

{{
  "technical": ["q1", "q2", "q3", "q4", "q5"],
  "project": ["q1", "q2", "q3"],
  "hr": ["q1", "q2", "q3"]
}}
"""
    raw = ask_llm(prompt).strip()
    print("RAW ROLE-BASED QUESTION RESPONSE:\n", raw)

    if raw.startswith("```"):
        raw = raw.replace("```json", "").replace("```", "").strip()

    try:
        data = json.loads(raw)
        return _normalize_questions(data)
    except Exception:
        return _fallback_questions(role)


def _normalize_questions(data):
    def normalize_list(items):
        cleaned = []
        for item in items:
            if isinstance(item, dict):
                cleaned.append(item.get("question", str(item)))
            else:
                cleaned.append(str(item))
        return cleaned

    return {
        "technical": normalize_list(data.get("technical", [])),
        "project": normalize_list(data.get("project", [])),
        "hr": normalize_list(data.get("hr", []))
    }


def _fallback_questions(role):
    return {
        "technical": [
            f"Explain a core data structure relevant to {role} and its time complexity.",
            f"How would you design a scalable system for a high-traffic application?",
            f"Write a function to find all pairs in an array that sum to a target value.",
            f"What is the difference between SQL joins? Explain with an example.",
            f"Explain OOP principles with a real-world example relevant to {role}."
        ],
        "project": [
            f"Describe your most complex project and your specific contribution.",
            f"What challenges did you face and how did you solve them?",
            f"How would you scale or improve your project if you had more time?"
        ],
        "hr": [
            "Tell me about yourself.",
            "What are your strengths and weaknesses?",
            "Why should we hire you for this role?"
        ]
    }

In [17]:
# ============================================================
# EVALUATE ANSWERS
# ============================================================

def evaluate_answer_llm(question, answer, round_name="technical"):
    prompt = f"""
You are an interview evaluator.

Interview Round: {round_name}
Question: {question}
Candidate Answer: {answer}

Evaluate the answer on: relevance, correctness, clarity, depth, confidence.
Give a score out of 10.

Return ONLY valid JSON:
{{
  "score": 0,
  "feedback": "",
  "improvement": "",
  "ideal_points": []
}}
"""
    raw = ask_llm(prompt).strip()
    if raw.startswith("```"):
        raw = raw.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(raw)
    except Exception:
        return {
            "score": 5,
            "feedback": "Average answer. Add more structure and technical detail.",
            "improvement": "Explain with better clarity and examples.",
            "ideal_points": []
        }

In [18]:
# ============================================================
# FINAL FEEDBACK GENERATION
# ============================================================

def generate_final_feedback_llm(profile, results):
    prompt = f"""
You are an expert interview coach.

Candidate Profile:
{json.dumps(profile, indent=2)}

Interview Results:
{json.dumps(results, indent=2)}

Create a final interview report.

Return ONLY valid JSON:
{{
  "overall_score": 0,
  "strengths": [],
  "weak_areas": [],
  "suggestions": [],
  "final_summary": ""
}}
"""
    raw = ask_llm(prompt).strip()
    if raw.startswith("```"):
        raw = raw.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(raw)
    except Exception:
        return {
            "overall_score": 0,
            "strengths": [],
            "weak_areas": [],
            "suggestions": [],
            "final_summary": "Final report generation failed."
        }


def format_question_text(q):
    if isinstance(q, dict):
        return q.get("question", str(q))
    return str(q)

In [19]:
# ============================================================
# INTERVIEW ENGINE
# ============================================================

class InterviewEngine:
    def __init__(self, profile, questions):
        self.profile = profile
        self.rounds = {
            "technical": questions.get("technical", []),
            "project": questions.get("project", []),
            "hr": questions.get("hr", [])
        }
        self.round_order = ["technical", "project", "hr"]
        self.current_round_index = 0
        self.current_round = self.round_order[self.current_round_index]
        self.current_index = 0
        self.results = {"technical": [], "project": [], "hr": []}

    def get_current_question(self):
        questions = self.rounds[self.current_round]
        if self.current_index < len(questions):
            return questions[self.current_index]
        return None

    def get_current_question_text(self):
        q = self.get_current_question()
        return format_question_text(q) if q is not None else None

    def submit_answer(self, answer):
        question_obj = self.get_current_question()
        if question_obj is None:
            return None
        question_text = format_question_text(question_obj)
        eval_result = evaluate_answer_llm(question_text, answer, self.current_round)
        self.results[self.current_round].append({
            "question": question_text,
            "answer": answer,
            "score": eval_result.get("score", 0),
            "feedback": eval_result.get("feedback", ""),
            "improvement": eval_result.get("improvement", ""),
            "ideal_points": eval_result.get("ideal_points", [])
        })
        self.current_index += 1
        return eval_result

    def move_to_next(self):
        if self.current_index < len(self.rounds[self.current_round]):
            return True
        self.current_round_index += 1
        if self.current_round_index < len(self.round_order):
            self.current_round = self.round_order[self.current_round_index]
            self.current_index = 0
            return True
        return False

In [20]:
# ============================================================
# GLOBAL STATE
# ============================================================

global_state = {
    "profile": None,
    "questions": None,
    "engine": None,
    "resume_text": None,
    "mode": None,
    "role": None,
    "company": None
}

In [21]:
# ============================================================
# PROCESS INTERVIEW — START
# ============================================================

def process_interview(mode, role, company, file):
    """
    Initialize interview based on mode (resume-based or role-based).
    Company and role now drive question difficulty and topic selection.
    """
    role = (role or "Software Engineer").strip()
    company = (company or "").strip()
    if company == "(Not specified)":
        company = ""

    company_type = COMPANY_TYPE_MAP.get(company, "unknown")
    company_info = f"{company} ({'Product-based' if company_type == 'product' else 'Service-based' if company_type == 'service' else 'General'})" if company else "Not specified"

    if mode == "Resume-based":
        if file is None:
            return "Please upload a resume.", "", ""
        raw_text = extract_resume_text(file.name)
        cleaned_text = clean_resume_text(raw_text)
        profile = build_candidate_profile_llm(cleaned_text)
        # Pass role and company so questions reflect both resume AND company/role expectations
        questions = generate_interview_questions(profile, role, company)
        global_state["resume_text"] = cleaned_text
    else:
        if not role:
            return "Please select a role.", "", ""
        questions = generate_role_company_questions(role, company)
        profile = {
            "name": "Practice Candidate",
            "summary": f"Role-based practice: {role} at {company_info}",
            "skills": [],
            "projects": [],
            "education": [],
            "experience": [],
            "target_roles": [role],
            "strengths": [],
            "company_target": company_info
        }
        global_state["resume_text"] = None

    engine = InterviewEngine(profile, questions)
    global_state.update({
        "profile": profile,
        "questions": questions,
        "engine": engine,
        "mode": mode,
        "role": role,
        "company": company
    })

    first_question = engine.get_current_question_text()
    status = f"Round: {engine.current_round.upper()} | Role: {role} | Company: {company_info} | Mode: {mode}"

    return (
        json.dumps(profile, indent=2),
        status,
        first_question if first_question else "No questions generated."
    )

In [22]:
# ============================================================
# SUBMIT ANSWER
# ============================================================

def submit_interview_answer(answer):
    engine = global_state["engine"]
    if engine is None:
        return "Please start the interview first.", "", "", ""
    if not answer or not answer.strip():
        return "Please write an answer.", f"Round: {engine.current_round.upper()}", engine.get_current_question_text(), ""

    result = engine.submit_answer(answer)
    if result is None:
        return "No active question found.", "", "", ""

    score_text = (
        f"Score: {result.get('score', 0)}/10\n\n"
        f"Feedback: {result.get('feedback', '')}\n\n"
        f"Improvement: {result.get('improvement', '')}"
    )

    next_question = engine.get_current_question_text()
    company_info = global_state.get("company") or "Not specified"
    role = global_state.get("role") or ""

    if next_question is not None:
        return score_text, f"Round: {engine.current_round.upper()} | Role: {role} | Company: {company_info}", next_question, ""

    moved = engine.move_to_next()
    if moved:
        next_question = engine.get_current_question_text()
        return score_text, f"Round: {engine.current_round.upper()} | Role: {role} | Company: {company_info}", next_question, ""

    return (
        score_text + "\n\nInterview Completed! Click 'Show Final Feedback' below.",
        "Interview Finished",
        "All rounds complete.",
        ""
    )

In [23]:
# ============================================================
# FINAL FEEDBACK + CHARTS
# ============================================================

def build_feedback_charts(results, final_report):
    round_scores = []
    for round_name in ["technical", "project", "hr"]:
        entries = results.get(round_name, [])
        avg_score = sum(item.get("score", 0) for item in entries) / len(entries) if entries else 0
        round_scores.append({"Round": round_name.capitalize(), "Average Score": round(avg_score, 2)})

    df_round = pd.DataFrame(round_scores)
    bar_fig = px.bar(
        df_round, x="Round", y="Average Score", color="Round", text="Average Score",
        title="Round-wise Interview Performance", range_y=[0, 10]
    )
    bar_fig.update_traces(textposition="outside")
    bar_fig.update_layout(template="plotly_dark", height=400, showlegend=False, title_x=0.5)

    donut_fig = go.Figure(data=[go.Pie(
        labels=["Strengths", "Weak Areas", "Suggestions"],
        values=[
            len(final_report.get("strengths", [])),
            len(final_report.get("weak_areas", [])),
            len(final_report.get("suggestions", []))
        ],
        hole=0.5
    )])
    donut_fig.update_layout(template="plotly_dark", height=400, title="Feedback Distribution", title_x=0.5)
    return bar_fig, donut_fig


def show_final_feedback():
    engine = global_state["engine"]
    profile = global_state["profile"]
    if engine is None or profile is None:
        return "No interview data found.", None, None

    final_report = generate_final_feedback_llm(profile, engine.results)
    formatted = (
        f"Overall Score: {final_report.get('overall_score', 0)}/10\n\n"
        f"Strengths:\n- " + "\n- ".join(final_report.get("strengths", [])) +
        f"\n\nWeak Areas:\n- " + "\n- ".join(final_report.get("weak_areas", [])) +
        f"\n\nSuggestions:\n- " + "\n- ".join(final_report.get("suggestions", [])) +
        f"\n\nFinal Summary:\n{final_report.get('final_summary', '')}"
    )
    bar_fig, donut_fig = build_feedback_charts(engine.results, final_report)
    return formatted, bar_fig, donut_fig


# ============================================================
# RESET
# ============================================================

def reset_all():
    global_state.update({
        "profile": None, "questions": None, "engine": None,
        "resume_text": None, "mode": None, "role": None, "company": None
    })
    return None, "", "", "", "", "", "", None, None

In [ ]:
# ============================================================
# GRADIO UI
# ============================================================

import gradio as gr

# ===== PREMIUM CSS =====
custom_css = """
body {
    background: #020617;
}

/* Container */
.gradio-container {
    background: radial-gradient(circle at top, #0f172a, #020617);
    color: #e2e8f0;
    font-family: 'Inter', sans-serif;
    padding: 10px 20px;
}

/* Header */
.header {
    text-align: center;
    margin-bottom: 20px;
}

.header-title {
    font-size: 2.4rem;
    font-weight: 800;
    color: #f8fafc;
}

.header-sub {
    color: #94a3b8;
    font-size: 0.95rem;
}

/* Card */
.card {
    background: rgba(15, 23, 42, 0.8);
    border-radius: 16px;
    padding: 16px;
    border: 1px solid rgba(255,255,255,0.08);
    box-shadow: 0 10px 30px rgba(0,0,0,0.4);
}

/* Section title */
.section-title {
    font-size: 0.95rem;
    color: #cbd5f5;
    margin-bottom: 8px;
    font-weight: 600;
}

/* Inputs */
textarea, input, select {
    background: #020617 !important;
    border: 1px solid #1e293b !important;
    color: #e2e8f0 !important;
    border-radius: 10px !important;
    padding: 10px !important;
}

/* Labels */
label {
    color: #c7d2fe !important;
    font-weight: 600 !important;
}

/* Buttons */
button {
    background: linear-gradient(135deg, #2563eb, #7c3aed) !important;
    border-radius: 10px !important;
    color: white !important;
    font-weight: 600 !important;
    border: none !important;
    transition: all 0.2s ease;
}

button:hover {
    transform: translateY(-1px);
    filter: brightness(1.1);
}

/* Question highlight */
.question-box textarea {
    background: rgba(59,130,246,0.1) !important;
    border: 1px solid #3b82f6 !important;
}

/* Output boxes */
textarea[readonly] {
    background: #010409 !important;
    border: 1px solid #1e293b !important;
}

/* Charts */
.plot-container {
    background: transparent !important;
}

/* Footer */
footer {
    display: none !important;
}
"""

theme = gr.themes.Soft()

with gr.Blocks(css=custom_css, theme=theme) as demo:

    # ===== HEADER =====
    gr.HTML("""
    <div class="header">
        <div class="header-title">Smart Interview Preparation</div>
        <div class="header-sub">Personalized mock interviews based on resume, role, and company</div>
    </div>
    """)

    with gr.Row():

        # ================= LEFT PANEL =================
        with gr.Column(scale=1):
            with gr.Group(elem_classes="card"):

                gr.Markdown("### Setup")

                mode_input = gr.Radio(
                    ["Resume-based"],
                    value="Resume-based",
                    label="Mode"
                )

                role_input = gr.Dropdown(
                    choices=ALL_ROLES,
                    value="Software Engineer",
                    label="Role",
                    allow_custom_value=True
                )

                company_input = gr.Dropdown(
                    choices=ALL_COMPANIES,
                    value="(Not specified)",
                    label="Company",
                    allow_custom_value=True
                )

                resume_input = gr.File(
                    label="Resume Upload"
                )

                with gr.Row():
                    upload_btn = gr.Button("Start")
                    clear_btn = gr.Button("Reset")

                profile_output = gr.Textbox(
                    label="Profile Summary",
                    lines=12,
                    interactive=False
                )

        # ================= RIGHT PANEL =================
        with gr.Column(scale=2):

            with gr.Group(elem_classes="card"):

                round_output = gr.Textbox(
                    label="Interview Status",
                    interactive=False
                )

                question_output = gr.Textbox(
                    label="Question",
                    lines=4,
                    interactive=False,
                    elem_classes="question-box"
                )

                answer_input = gr.Textbox(
                    label="Your Answer",
                    lines=6,
                    placeholder="Write your answer clearly and structured"
                )

                with gr.Row():
                    submit_btn = gr.Button("Submit Answer")
                    final_btn = gr.Button("Final Feedback")

                score_output = gr.Textbox(
                    label="Evaluation",
                    lines=6,
                    interactive=False
                )

    # ================= ANALYTICS =================
    with gr.Row():

        with gr.Column():
            with gr.Group(elem_classes="card"):
                performance_chart = gr.Plot(label="Performance")

        with gr.Column():
            with gr.Group(elem_classes="card"):
                feedback_chart = gr.Plot(label="Feedback Insights")

    with gr.Group(elem_classes="card"):
        final_report_output = gr.Textbox(
            label="Final Report",
            lines=16,
            interactive=False
        )

    # ===== LOGIC  =====
    upload_btn.click(
        fn=process_interview,
        inputs=[mode_input, role_input, company_input, resume_input],
        outputs=[profile_output, round_output, question_output]
    )

    submit_btn.click(
        fn=submit_interview_answer,
        inputs=[answer_input],
        outputs=[score_output, round_output, question_output, answer_input]
    )

    final_btn.click(
        fn=show_final_feedback,
        inputs=[],
        outputs=[final_report_output, performance_chart, feedback_chart]
    )

    clear_btn.click(
        fn=reset_all,
        inputs=[],
        outputs=[
            resume_input, profile_output, round_output, question_output,
            answer_input, score_output, final_report_output,
            performance_chart, feedback_chart
        ]
    )

# RUN
demo.launch(share=True, debug=True)

/tmp/ipykernel_3878/1856463387.py:110: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, theme=theme) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bc145fae4e5b26078e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
